# 01 数据总览

用 quant.query 看看数据库里有什么。运行前确保已经跑过 quant db-migrate / quant update-*。

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from quant import query

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

## 数据概况

In [ ]:
info = query.summary()
for key in ("price", "fundamental"):
    print(key, "->", info[key])
print()
for table, rows in info["tables"].items():
    print("%-24s%12d" % (table, rows))

## 全市场快照

snapshot() 返回「一行一只股票」的横截面表：最新行情 + 最近已披露季度 + 最新财务指标 + 派生估值。

In [ ]:
snap = query.snapshot()
print(snap.shape)
snap[["symbol", "name", "date", "close_org", "close_qfq", "season",
      "total_revenue", "season_income_growth", "market_cap", "pe", "pb"]].head(10)

## 市值分布

In [ ]:
cap_yi = (snap["market_cap"] / 1e8).dropna()
print("有市值的股票:", len(cap_yi), "/", len(snap))

fig, ax = plt.subplots(figsize=(10, 4))
cap_yi[cap_yi < 2000].hist(bins=60, ax=ax)
ax.set_xlabel("总市值（亿元）")
ax.set_ylabel("股票数")
ax.set_title("全市场市值分布（<2000 亿）")
plt.show()

## 单只股票的日线与估值

In [ ]:
SYMBOL = "000001"

price = query.prices(SYMBOL, start=20180101)
print(price[["date", "open", "close", "volume"]].tail(3))

In [ ]:
from quant.price.loader import get_full_price

full = get_full_price(SYMBOL)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
full["close_org"].plot(ax=axes[0], title=SYMBOL + " 收盘价（不复权）")
full["PE"].replace(0, float("nan")).plot(ax=axes[1], title="PE(TTM)")
plt.tight_layout()
plt.show()

## 基本面时序

growth() 返回单季收入 / 利润及其同比。

In [ ]:
growth = query.growth(symbols=[SYMBOL])
growth[["season", "season_income", "season_income_growth",
        "season_profit", "season_profit_growth"]].tail(12)